
# Telco Customer Churn — Automated Data Validation

## Objective

This notebook implements automated data quality
validation using Great Expectations (GX).

The goal is to convert the findings from the
initial data audit into reusable validation rules.

The validation focuses on:
- Dataset structure
- Customer identifier integrity
- Valid categorical values
- Numerical constraints
- Invalid TotalCharges values

The original dataset remains unchanged.


In [1]:
from pathlib import Path

import pandas as pd
import great_expectations as gx

# Project root
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Load raw data
data_path = (
    project_root / "data" / "raw"
    / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df = pd.read_csv(data_path)

print("Great Expectations version:", gx.__version__)
print("Dataset shape:", df.shape)


Great Expectations version: 1.25.0
Dataset shape: (7043, 21)



## 1. Initialize Great Expectations

A Data Context is the central entry point for
managing Great Expectations components.

We use an Ephemeral Data Context for experimentation.
It stores GX configurations in memory without
creating persistent configuration files.


In [2]:
context = gx.get_context(mode="ephemeral")

print(type(context).__name__)


EphemeralDataContext



## 2. Connect to Data

We register the Pandas DataFrame as a GX Data Asset.

The entire dataset will be treated as a single
Batch for validation.


In [3]:
# 1. Create a Pandas Data Source
data_source = context.data_sources.add_pandas(
    name="telco_source"
)

# 2. Register a Data Asset
data_asset = data_source.add_dataframe_asset(
    name="telco_customers"
)

# 3. Define how to retrieve the data
batch_definition = (
    data_asset.add_batch_definition_whole_dataframe(
        "full_dataset"
    )
)

# 4. Create the Batch
batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df}
)

print("Batch created successfully")


Batch created successfully



## 3. Define and Test Individual Expectations

An Expectation defines a data quality requirement.

We first test a single rule to understand how
GX evaluates data and reports validation results.


In [4]:
expectation = (
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="customerID"
    )
)

result = batch.validate(expectation)

print("Validation passed:", result.success)
print("Unexpected values:", result.result.get("unexpected_count"))


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Validation passed: True
Unexpected values: 0



## 4. Create an Expectation Suite

An Expectation Suite is a collection of data
quality rules applied to the same dataset.

We define a baseline validation suite for the
raw Telco Customer Churn dataset.


The suite checks:
- Dataset schema integrity (21 expected columns)
- Customer ID completeness and uniqueness
- Churn label completeness and validity
- Valid contract categories
- Non-negative tenure and monthly charges

The known TotalCharges issue is validated separately.



In [5]:

# Create a suite
suite = gx.ExpectationSuite(
    name="telco_raw_quality"
)

# Customer IDs must be present
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="customerID"
    )
)

# Customer IDs must be unique
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="customerID"
    )
)

# Churn must contain valid categories
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="Churn",
        value_set=["Yes", "No"]
    )
)


suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="Churn"
    )
)


# Contract categories must be valid
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="Contract",
        value_set=[
            "Month-to-month",
            "One year",
            "Two year"
        ]
    )
)

# Tenure must be non-negative
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="tenure",
        min_value=0
    )
)

# Monthly charges must be non-negative
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="MonthlyCharges",
        min_value=0
    )
)


expected_columns = [
    "customerID", "gender", "SeniorCitizen",
    "Partner", "Dependents", "tenure",
    "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
    "MonthlyCharges", "TotalCharges", "Churn"
]

suite.add_expectation(
    gx.expectations.ExpectTableColumnsToMatchSet(
        column_set=expected_columns,
        exact_match=True
    )
)


suite = context.suites.add_or_update(suite)

print("Expectation Suite created")
print("Rules:", len(suite.expectations))


Expectation Suite created
Rules: 8



## 5. Execute the Validation Suite

A Validation Definition connects a Batch Definition
to an Expectation Suite.

This allows multiple data quality rules to be
evaluated together and produces structured
validation results.


In [6]:

validation_definition = gx.ValidationDefinition(
    name="telco_raw_validation",
    data=batch_definition,
    suite=suite
)


validation_definition = (
    context.validation_definitions.add_or_update(
        validation_definition
    )
)


validation_result = validation_definition.run(
    batch_parameters={"dataframe": df}
)

print("Overall success:", validation_result.success)


Calculating Metrics:   0%|          | 0/44 [00:00<?, ?it/s]

Overall success: True


In [7]:

validation_summary = pd.DataFrame([
    {
        "expectation": r.expectation_config.type,
        "column": r.expectation_config.kwargs.get("column", "N/A"),
        "passed": r.success,
        "unexpected_count": r.result.get("unexpected_count", 0)
    }
    for r in validation_result.results
])

display(validation_summary)


,expectation,column,passed,unexpected_count
0,expect_column_values_to_not_be_null,customerID,True,0
1,expect_column_values_to_be_unique,customerID,True,0
2,expect_column_values_to_be_in_set,Churn,True,0
3,expect_column_values_to_not_be_null,Churn,True,0
4,expect_column_values_to_be_in_set,Contract,True,0
5,expect_column_values_to_be_between,tenure,True,0
6,expect_column_values_to_be_between,MonthlyCharges,True,0
7,expect_table_columns_to_match_set,N/A,True,0



## 6. Detect Invalid TotalCharges Values

The initial audit identified 11 blank entries
in `TotalCharges`.

We use a regular expression Expectation to verify
that each value represents a non-negative number.

This validation is expected to fail on the raw
dataset because the known invalid values have
not yet been corrected.


In [8]:

total_charges_expectation = (
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column="TotalCharges",
        regex=r"^\s*\d+(?:\.\d+)?\s*$"
    )
)

total_charges_result = batch.validate(
    total_charges_expectation
)

print("Validation passed:", total_charges_result.success)

print(
    "Invalid values:",
    total_charges_result.result.get("unexpected_count")
)


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Validation passed: False
Invalid values: 11



### 6.1 Validation Robustness Test

To verify that the validation suite correctly
identifies unexpected data, we intentionally
introduce an invalid churn category.

The validation is expected to fail because
`Unknown` is not an authorized churn label.

The test is performed on a copy of the dataset,
preserving the original observations.


In [9]:

# Create a copy: never modify the raw dataset
df_test = df.copy()

# Introduce an invalid churn category
df_test.loc[0, "Churn"] = "Unknown"

# Re-run the same validation suite
test_result = validation_definition.run(
    batch_parameters={"dataframe": df_test}
)

print("Validation passed:", test_result.success)

for r in test_result.results:
    if not r.success:
        print(
            r.expectation_config.type,
            r.expectation_config.kwargs.get("column"),
            r.result.get("unexpected_count")
        )


# Verify expected validation behavior
assert validation_result.success, (
    "Baseline validation should pass."
)

assert not total_charges_result.success, (
    "Invalid TotalCharges should be detected."
)

assert (
    total_charges_result.result["unexpected_count"] == 11
), "Expected 11 invalid TotalCharges entries."


assert any(
    not r.success
    and r.expectation_config.kwargs.get("column") == "Churn"
    for r in test_result.results
), "Invalid Churn category was not detected."


print("All GX validation tests passed.")



Calculating Metrics:   0%|          | 0/44 [00:00<?, ?it/s]

Validation passed: False
expect_column_values_to_be_in_set Churn 1
All GX validation tests passed.



## 7. Validation Summary

Great Expectations was successfully integrated
into the data quality workflow.

### Implemented Validations
- Customer ID completeness
- Customer ID uniqueness
- Valid churn labels
- Valid contract categories
- Non-negative tenure
- Non-negative monthly charges
- Detection of invalid TotalCharges entries


### Key Findings

- The baseline data validation suite passed.
- 11 invalid `TotalCharges` entries were identified.
- These invalid values represent a known raw-data
  quality issue requiring preprocessing.
- An intentionally corrupted churn label was
  successfully detected by GX.
- The raw dataset was preserved without modification.

### Validation Strategy

The baseline suite verifies structural and essential
data quality requirements.

The `TotalCharges` validation is performed separately
to demonstrate the detection of known anomalies.

A stricter validation suite will be introduced after
preprocessing to ensure model-ready data quality.


### Next Steps
- Integrate GX validation into a reusable Python module.
- Define separate validation suites for raw and
  processed datasets.
- Integrate automated quality checks into the
  preprocessing pipeline.
- Use persistent GX configurations and Checkpoints
  for production-oriented validation.
